# PROYEK DATA SCIENCE & MACHINE LEARNING
## Prediksi Debit Aliran Sungai dan Risiko Banjir Berbasis Curah Hujan Menggunakan Gradient Boosting Regressor

> **Mata Kuliah:** Data Science & Machine Learning  
> **Lokasi Pengamatan:** 22 kota di Indonesia yang punya data debit dan hujan (dari 24 lokasi pada `locations.csv`; Denpasar dan Surabaya tidak punya data debit)  
> **Periode Data:** ±1 tahun harian (12 Okt 2025 s.d. 9 Okt 2026 pada irisan debit dan hujan)  
> **Sumber Data:** `locations.csv` (Open-Meteo Geocoding), `river_discharge.csv` (Open-Meteo Flood API / GloFAS), `rainfall_daily.csv` (NASA POWER)  
> **Algoritma Utama:** Gradient Boosting Regressor (GBR)  
> **Model Pembanding:** Baseline Persistence (debit besok = debit hari ini)  
> **Ambang Siaga:** Persentil ke-90 ($P_{90}$) debit per kota, dihitung dari data latih saja

---

## Lima Pertanyaan Bisnis

| No | Tipe | Fokus | Pertanyaan Bisnis |
|:---|:---|:---|:---|
| **1** | **Dasar** | Gambaran umum data | Bagaimana gambaran umum data dan karakter debit sungai serta curah hujan di tiap kota rawan banjir? |
| **2** | **Analitik (masalah)** | Tingkat risiko | Kota mana yang paling berisiko banjir, dilihat dari seberapa ekstrem puncak debit dan seberapa sering debit melonjak? |
| **3** | **Analitik (urgensi)** | Waktu peringatan | Seberapa cepat debit naik setelah hujan lebat, sehingga berapa hari waktu peringatan dini yang tersedia? |
| **4** | **Analitik (pola dan pemicu)** | Musiman dan korelasi | Bagaimana pola musiman debit, bulan apa yang paling berisiko, dan seberapa kuat hubungan hujan dengan debit? |
| **5** | **AI (metode, algoritma)** | Akurasi dan keandalan model | Algoritma apa yang paling akurat memprediksi debit 1 hari ke depan, fitur apa yang paling berpengaruh, dan apakah model andal mendeteksi kondisi di atas ambang siaga? |

---

## TAHAP 1: Import Library

Satu sel untuk semua library yang dibutuhkan sampai akhir proyek: pengolahan data (pandas, numpy), visualisasi (matplotlib, seaborn), statistik (scipy), model dan evaluasi (scikit-learn), dan penyimpanan model (joblib, json, zipfile).

In [ ]:
!pip install -q pandas numpy matplotlib seaborn scipy scikit-learn joblib

In [ ]:
import os
import json
import warnings
import zipfile
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             classification_report, confusion_matrix)
import joblib

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

print("Seluruh library berhasil diimpor!")

## TAHAP 2: Data Assessing dan Data Cleaning (3 Tabel Terpisah)

### 2.1 Memuat 3 Tabel

Kode mencari file CSV di beberapa folder (Google Drive, folder lokal, Downloads). Nama kolom debit, hujan, dan kota dideteksi otomatis. Jika hasil deteksi salah, ubah manual tiga variabel `COL_...` di bawah.

In [ ]:
search_dirs = [
    '/content/drive/MyDrive',
    '/content/drive/MyDrive/flood_data',
    '/content/drive/MyDrive/debit_sungai',
    r'C:\Users\reno\Downloads',
    '.', '../'
]

def locate_csv(filename):
    for d in search_dirs:
        target = os.path.join(d, filename)
        if os.path.exists(target):
            return target
    return filename

df_loc_raw  = pd.read_csv(locate_csv('locations.csv'))
df_dis_raw  = pd.read_csv(locate_csv('river_discharge.csv'))
df_rain_raw = pd.read_csv(locate_csv('rainfall_daily.csv'))

# --- Deteksi nama kolom (ubah manual bila perlu) ---
COL_DIS = 'river_discharge'
COL_RAIN = 'rain_mm'
COL_CITY = None

if COL_DIS not in df_dis_raw.columns:
    COL_DIS = [c for c in df_dis_raw.select_dtypes('number').columns if c != 'location_id'][0]
if COL_RAIN not in df_rain_raw.columns:
    COL_RAIN = [c for c in df_rain_raw.select_dtypes('number').columns if c != 'location_id'][0]
for c in ['city', 'city_name', 'name', 'location_name', 'kota']:
    if c in df_loc_raw.columns:
        COL_CITY = c
        break
if COL_CITY is None:
    COL_CITY = [c for c in df_loc_raw.select_dtypes('object').columns][0]

print(f"Kolom debit: {COL_DIS} | Kolom hujan: {COL_RAIN} | Kolom kota: {COL_CITY}")
print(f"Locations: {df_loc_raw.shape} | Discharge: {df_dis_raw.shape} | Rainfall: {df_rain_raw.shape}")

### 2.2 Assessing: `head`, `info`, `describe`, Missing Value, Duplikat

Fungsi `assess()` dipakai berulang untuk tiap tabel. Ia menampilkan 5 baris awal, tipe data (`info`), statistik ringkas (`describe`), jumlah nilai kosong, baris duplikat penuh, duplikat pada kunci (`location_id` + `obs_date`), dan nilai sentinel -999 (kode data hilang NASA POWER).

In [ ]:
def assess(df, nama, key_cols=None):
    print("=" * 70)
    print(f"ASSESSING: {nama}")
    print("=" * 70)
    print("\n[1] 5 baris pertama"); display(df.head())
    print("\n[2] df.info()"); df.info()
    print("\n[3] df.describe()"); display(df.describe(include='all').T)
    print("\n[4] Missing value per kolom"); print(df.isna().sum())
    print(f"\n[5] Duplikat baris penuh : {df.duplicated().sum()}")
    if key_cols:
        print(f"[6] Duplikat kunci {key_cols} : {df.duplicated(subset=key_cols).sum()}")
    num = df.select_dtypes('number')
    print(f"[7] Nilai sentinel -999  : {(num == -999).sum().sum()}")
    print(f"[8] Nilai negatif        : {(num < 0).sum().sum()}\n")

assess(df_loc_raw,  'LOCATIONS',       ['location_id'])
assess(df_dis_raw,  'RIVER DISCHARGE', ['location_id', 'obs_date'])
assess(df_rain_raw, 'RAINFALL DAILY',  ['location_id', 'obs_date'])

### 2.3 Cleaning

Perbaikan berdasarkan temuan assessing:
1. Kolom tanggal diubah ke tipe `datetime`.
2. Nilai -999 dan debit negatif dianggap data hilang (`NaN`).
3. Data hilang diisi interpolasi linear **per kota** (tidak lintas kota), sisa di ujung deret diisi `bfill`/`ffill`.
4. Data diurutkan kronologis dan duplikat dibuang.
5. Teks lokasi diseragamkan: awalan "Kota " dan "DI " pada nama kota dihapus, serta awalan "Provinsi", "Propinsi", "Daerah Tingkat I" pada nama provinsi dihapus (ditemukan tidak konsisten saat assessing `locations`).

In [ ]:
df_loc  = df_loc_raw.copy()
df_dis  = df_dis_raw.copy()
df_rain = df_rain_raw.copy()

# 1. Tanggal
df_dis['obs_date']  = pd.to_datetime(df_dis['obs_date'])
df_rain['obs_date'] = pd.to_datetime(df_rain['obs_date'])

# 2. Sentinel dan nilai tidak wajar menjadi NaN
for col in df_rain.select_dtypes('number').columns:
    if col != 'location_id':
        df_rain.loc[df_rain[col] == -999.0, col] = np.nan
df_rain.loc[df_rain[COL_RAIN] < 0, COL_RAIN] = np.nan
df_dis.loc[df_dis[COL_DIS] < 0, COL_DIS] = np.nan

# 3. Drop duplikat dan urutkan
df_dis  = df_dis.sort_values(['location_id', 'obs_date']).drop_duplicates(['location_id', 'obs_date'])
df_rain = df_rain.sort_values(['location_id', 'obs_date']).drop_duplicates(['location_id', 'obs_date'])
df_loc  = df_loc.drop_duplicates('location_id')

# 4. Isi data hilang per kota
fill = lambda g: g.interpolate().bfill().ffill()
df_dis[COL_DIS]   = df_dis.groupby('location_id')[COL_DIS].transform(fill)
df_rain[COL_RAIN] = df_rain.groupby('location_id')[COL_RAIN].transform(fill)

# 5. Standarisasi teks lokasi
df_loc[COL_CITY] = (df_loc[COL_CITY].str.strip()
                    .str.replace(r'^(Kota|Kabupaten)\s+', '', regex=True)
                    .str.replace(r'^DI\s+', '', regex=True))
if 'province' in df_loc.columns:
    df_loc['province'] = (df_loc['province'].str.strip()
                          .str.replace(r'^(Provinsi|Propinsi|Daerah Tingkat I)\s+', '', regex=True))

print("Cleaning selesai.")
print("Sisa missing debit :", df_dis[COL_DIS].isna().sum())
print("Sisa missing hujan :", df_rain[COL_RAIN].isna().sum())

## TAHAP 3: Penggabungan 3 Tabel (Merge) dan Assessing Ulang

### 3.1 Merge

`river_discharge` digabung dengan `rainfall_daily` lewat kunci (`location_id`, `obs_date`) memakai `inner` join agar hanya tanggal yang punya data debit **dan** hujan yang dipakai. Hasilnya digabung dengan `locations` (`left` join) untuk memperoleh nama kota. Kolom lalu diseragamkan menjadi `kota`, `debit`, `hujan`.

In [ ]:
df = pd.merge(df_dis, df_rain, on=['location_id', 'obs_date'], how='inner', suffixes=('', '_rain'))
df = pd.merge(df, df_loc, on='location_id', how='left', suffixes=('', '_loc'))
df = df.rename(columns={COL_DIS: 'debit', COL_RAIN: 'hujan', COL_CITY: 'kota'})
df = df.sort_values(['kota', 'obs_date']).reset_index(drop=True)

print("Shape hasil gabungan:", df.shape)
display(df.head())

### 3.2 Assessing Ulang Data Gabungan

Pemeriksaan ulang setelah merge: missing, duplikat, kota tanpa nama (kunci tidak cocok), lokasi yang tidak punya data debit, cakupan tanggal per kota, dan tanggal yang bolong (penting karena fitur lag mengasumsikan data harian berurutan). Perhatikan juga bahwa data hujan berakhir lebih awal daripada data debit, sehingga `inner` join otomatis memotong beberapa hari terakhir debit.

In [ ]:
assess(df, 'DATA GABUNGAN', ['kota', 'obs_date'])

print("Baris tanpa nama kota (join gagal):", df['kota'].isna().sum())

tanpa_debit = set(df_loc['location_id']) - set(df_dis['location_id'])
print("Lokasi di locations.csv tanpa data debit (tidak ikut analisis):", tanpa_debit)
print("Irisan tanggal debit dan hujan:", df['obs_date'].min().date(), "s.d.", df['obs_date'].max().date())

cakupan = df.groupby('kota')['obs_date'].agg(['min', 'max', 'count'])
cakupan['hari_seharusnya'] = (cakupan['max'] - cakupan['min']).dt.days + 1
cakupan['tanggal_bolong'] = cakupan['hari_seharusnya'] - cakupan['count']
print("\nCakupan tanggal per kota:")
display(cakupan)

df = df.dropna(subset=['kota', 'debit', 'hujan']).drop_duplicates(['kota', 'obs_date'])
print("Shape final setelah dibersihkan:", df.shape)

## TAHAP 4: Exploratory Data Analysis (4 Grafik, Menjawab Pertanyaan 1 sampai 4)

### 4.1 Grafik 1: Gambaran Umum Debit dan Hujan (Pertanyaan 1)

Boxplot debit per kota (skala log karena nilai debit sangat timpang antar sungai) dan batang rata-rata curah hujan harian. Jawaban dicetak otomatis di bawahnya.

In [ ]:
urut = df.groupby('kota')['debit'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(1, 2, figsize=(16, 5.5))
sns.boxplot(data=df, x='kota', y='debit', order=urut, ax=ax[0], palette='Blues')
ax[0].set_yscale('log')
ax[0].set_title('Sebaran Debit Sungai per Kota (skala log)')
ax[0].set_ylabel('Debit (m³/s)'); ax[0].set_xlabel('')
ax[0].tick_params(axis='x', rotation=60)

hujan_rata = df.groupby('kota')['hujan'].mean().reindex(urut)
sns.barplot(x=hujan_rata.index, y=hujan_rata.values, ax=ax[1], palette='crest')
ax[1].set_title('Rata-rata Curah Hujan Harian per Kota')
ax[1].set_ylabel('Hujan (mm/hari)'); ax[1].set_xlabel('')
ax[1].tick_params(axis='x', rotation=60)
plt.tight_layout(); plt.show()

ringkas = df.groupby('kota').agg(
    n_hari=('debit', 'count'),
    debit_median=('debit', 'median'),
    debit_maks=('debit', 'max'),
    hujan_rata=('hujan', 'mean'),
    hari_hujan_pct=('hujan', lambda s: (s >= 1).mean() * 100)
).round(2)
display(ringkas)

print("\n[JAWABAN 1]")
print(f"- Data: {df['kota'].nunique()} kota, {df['obs_date'].min().date()} s.d. {df['obs_date'].max().date()}, {len(df):,} baris.")
print(f"- Debit median tertinggi : {ringkas['debit_median'].idxmax()} ({ringkas['debit_median'].max():.1f} m³/s)")
print(f"- Hujan rata-rata tertinggi : {ringkas['hujan_rata'].idxmax()} ({ringkas['hujan_rata'].max():.2f} mm/hari)")

### 4.2 Grafik 2: Tingkat Risiko Kota (Pertanyaan 2)

Dua ukuran risiko: **rasio puncak** (debit maksimum dibagi median, makin besar makin ekstrem) dan **frekuensi lonjakan** (persen hari ketika debit naik 20% atau lebih dibanding kemarin). Kota dengan rata-rata peringkat terbaik pada kedua ukuran dinilai paling berisiko.

In [ ]:
d = df.sort_values(['kota', 'obs_date']).copy()
d['naik_pct'] = d.groupby('kota')['debit'].pct_change().replace([np.inf, -np.inf], np.nan)  # inf muncul bila debit kemarin = 0

risiko = d.groupby('kota').agg(med=('debit', 'median'), maks=('debit', 'max'))
risiko['rasio_puncak'] = risiko['maks'] / risiko['med'].clip(lower=1e-6)
risiko['frek_lonjakan_pct'] = d.groupby('kota')['naik_pct'].apply(lambda s: (s >= 0.20).mean() * 100)
risiko['skor_peringkat'] = risiko[['rasio_puncak', 'frek_lonjakan_pct']].rank(ascending=False).mean(axis=1)
risiko = risiko.sort_values('skor_peringkat').round(2)

fig, ax = plt.subplots(1, 2, figsize=(16, 5.5))
sns.barplot(x=risiko.index, y=risiko['rasio_puncak'], ax=ax[0], palette='Reds_r')
ax[0].set_title('Rasio Puncak Debit (Maks / Median)'); ax[0].set_xlabel('')
ax[0].tick_params(axis='x', rotation=60)
sns.barplot(x=risiko.index, y=risiko['frek_lonjakan_pct'], ax=ax[1], palette='Oranges_r')
ax[1].set_title('Frekuensi Lonjakan Debit (≥ 20% per hari)'); ax[1].set_xlabel('')
ax[1].set_ylabel('% hari')
ax[1].tick_params(axis='x', rotation=60)
plt.tight_layout(); plt.show()

display(risiko)
print("\n[JAWABAN 2]")
print(f"- Kota paling berisiko: {risiko.index[0]} (rasio puncak {risiko.iloc[0]['rasio_puncak']}x, lonjakan {risiko.iloc[0]['frek_lonjakan_pct']}% hari)")
print(f"- Urutan risiko tinggi ke rendah: {', '.join(risiko.index)}")

### 4.3 Grafik 3: Kecepatan Respons Debit terhadap Hujan (Pertanyaan 3)

Untuk setiap kota dihitung korelasi antara **kenaikan debit hari ini** dan **curah hujan N hari sebelumnya** (N = 0 sampai 7). Karena ada 22 kota, hasilnya ditampilkan sebagai heatmap (baris = kota, kolom = lag). Kotak dengan warna paling pekat pada satu baris menunjukkan berapa hari setelah hujan debit paling kuat naik, yang menjadi perkiraan waktu peringatan dini.

In [ ]:
lags = list(range(0, 8))
cc = {}
for kota, g in df.sort_values('obs_date').groupby('kota'):
    kenaikan = g['debit'].diff()
    cc[kota] = [kenaikan.corr(g['hujan'].shift(l)) for l in lags]
cc = pd.DataFrame(cc, index=lags)

plt.figure(figsize=(9, 9))
sns.heatmap(cc.T, annot=True, fmt='.2f', cmap='YlGnBu', cbar_kws={'label': 'Korelasi'})
plt.title('Korelasi Kenaikan Debit dengan Hujan N Hari Sebelumnya')
plt.xlabel('Lag hujan (hari sebelum)'); plt.ylabel('')
plt.tight_layout(); plt.show()

lag_terbaik = cc.idxmax()
print("[JAWABAN 3] Waktu peringatan dini (lag korelasi tertinggi):")
for kota, l in lag_terbaik.items():
    teks = "kurang dari 1 hari (respons pada hari yang sama)" if l == 0 else f"sekitar {l} hari setelah hujan"
    print(f"- {kota}: {teks} (korelasi {cc.loc[l, kota]:.2f})")
print(f"- Rata-rata semua kota: {lag_terbaik.mean():.1f} hari | median: {lag_terbaik.median():.0f} hari")

### 4.4 Grafik 4: Pola Musiman dan Hubungan Hujan–Debit (Pertanyaan 4)

Heatmap rata-rata debit per bulan, dinormalisasi per kota (dibagi rata-rata debit kota itu) sehingga semua kota bisa dibandingkan. Nilai di atas 1 berarti debit bulan itu di atas rata-rata tahunan. Korelasi hujan dan debit dihitung terpisah.

In [ ]:
df['bulan'] = df['obs_date'].dt.month
pivot = df.groupby(['kota', 'bulan'])['debit'].mean().unstack()
pivot_norm = pivot.div(pivot.mean(axis=1), axis=0)

plt.figure(figsize=(13, 9))
sns.heatmap(pivot_norm, annot=True, fmt='.2f', cmap='YlOrRd', cbar_kws={'label': 'Debit relatif (1 = rata-rata)'})
plt.title('Pola Musiman Debit per Kota (dinormalisasi)')
plt.xlabel('Bulan'); plt.ylabel('')
plt.show()

df = df.sort_values(['kota', 'obs_date'])
df['hujan_sum3'] = df.groupby('kota')['hujan'].transform(lambda s: s.rolling(3).sum())
korelasi = df.groupby('kota').apply(lambda g: pd.Series({
    'pearson_hujan_vs_debit': g['hujan'].corr(g['debit']),
    'pearson_hujan3hari_vs_debit': g['hujan_sum3'].corr(g['debit']),
    'spearman_hujan3hari_vs_debit': g['hujan_sum3'].corr(g['debit'], method='spearman')
})).round(3)
display(korelasi)

bulan_puncak = pivot_norm.mean().idxmax()
print("\n[JAWABAN 4]")
print(f"- Bulan paling berisiko (semua kota): bulan {bulan_puncak} (debit relatif {pivot_norm.mean().max():.2f}x rata-rata)")
for kota, b in pivot_norm.idxmax(axis=1).items():
    print(f"  - {kota}: puncak di bulan {b}")
print(f"- Rata-rata korelasi Spearman hujan 3 hari vs debit: {korelasi['spearman_hujan3hari_vs_debit'].mean():.2f}")
df = df.drop(columns=['hujan_sum3'])

## TAHAP 5: Feature Engineering dan Pembagian Data

### 5.1 Membuat Fitur

Model memprediksi **debit besok**. Skala debit antar kota sangat berbeda (median dari sekitar 1 m³/s sampai lebih dari 500 m³/s), sehingga model yang belajar langsung dari angka debit akan didominasi sungai besar. Karena itu model dilatih memprediksi **perubahan relatif** (selisih log debit besok terhadap debit hari ini), lalu hasilnya diubah kembali menjadi m³/s. Semua fitur hanya memakai informasi sampai hari ini:
- **Debit (skala log)**: log debit hari ini, serta perubahan dibanding 1, 3, 7 hari lalu dan dibanding rata-rata 7 hari.
- **Hujan**: hari ini, lag 1, 2, 3 hari, total 3 hari, dan total 7 hari.
- **Musiman**: bulan dalam bentuk sin/cos agar Desember dan Januari dianggap berdekatan.
- **Kota**: one-hot (kolom 0/1 per kota).

Fungsi `buat_fitur()` dibuat sebagai fungsi tersendiri karena dashboard nanti memakai logika yang sama persis.

In [ ]:
def buat_fitur(data, dengan_target=True):
    d = data.sort_values(['kota', 'obs_date']).copy()
    g = d.groupby('kota')

    # Debit
    d['debit_t0'] = d['debit']
    for l in (1, 2, 3, 7):
        d[f'debit_lag{l}'] = g['debit'].shift(l)
    d['debit_mean7'] = g['debit'].transform(lambda s: s.rolling(7).mean())
    d['log_debit_t0'] = np.log1p(d['debit_t0'])
    d['dlog_1'] = d['log_debit_t0'] - np.log1p(d['debit_lag1'])
    d['dlog_3'] = d['log_debit_t0'] - np.log1p(d['debit_lag3'])
    d['dlog_7'] = d['log_debit_t0'] - np.log1p(d['debit_lag7'])
    d['dlog_mean7'] = d['log_debit_t0'] - np.log1p(d['debit_mean7'])

    # Hujan
    d['hujan_t0'] = d['hujan']
    for l in (1, 2, 3):
        d[f'hujan_lag{l}'] = g['hujan'].shift(l)
    d['hujan_sum3'] = g['hujan'].transform(lambda s: s.rolling(3).sum())
    d['hujan_sum7'] = g['hujan'].transform(lambda s: s.rolling(7).sum())

    # Musiman
    bulan = d['obs_date'].dt.month
    d['bulan_sin'] = np.sin(2 * np.pi * bulan / 12)
    d['bulan_cos'] = np.cos(2 * np.pi * bulan / 12)

    # Target: debit besok (asli) dan perubahan log-nya (yang dipelajari model)
    if dengan_target:
        d['target_debit_besok'] = g['debit'].shift(-1)
        d['target_dlog'] = np.log1p(d['target_debit_besok']) - d['log_debit_t0']
    return d

FITUR_NUMERIK = ['log_debit_t0', 'dlog_1', 'dlog_3', 'dlog_7', 'dlog_mean7',
                 'hujan_t0', 'hujan_lag1', 'hujan_lag2', 'hujan_lag3', 'hujan_sum3', 'hujan_sum7',
                 'bulan_sin', 'bulan_cos']

df_fe = buat_fitur(df)
dummies = pd.get_dummies(df_fe['kota'], prefix='kota').astype(int)
df_fe = pd.concat([df_fe, dummies], axis=1)

FITUR = FITUR_NUMERIK + list(dummies.columns)
df_fe = df_fe.dropna(subset=FITUR + ['target_debit_besok']).reset_index(drop=True)

print("Jumlah fitur:", len(FITUR))
print("Shape data model:", df_fe.shape)
display(df_fe[FITUR + ['target_dlog', 'target_debit_besok']].head())

### 5.2 Split Data Latih dan Uji (Berdasarkan Waktu)

Deret waktu **tidak boleh diacak**, karena model tidak boleh "melihat masa depan". Data dipotong pada tanggal tertentu: 80% tanggal paling awal untuk latih, 20% terbaru untuk uji, berlaku untuk semua kota sekaligus. Karena data hanya sekitar 1 tahun, data uji (±2 bulan terakhir) umumnya berisi musim kering sehingga kejadian di atas ambang siaga sedikit; baca hasil deteksi siaga dengan hati-hati.

In [ ]:
tgl_potong = df_fe['obs_date'].sort_values().iloc[int(0.8 * len(df_fe))]

df_train = df_fe[df_fe['obs_date'] < tgl_potong].sort_values('obs_date').reset_index(drop=True)
df_test  = df_fe[df_fe['obs_date'] >= tgl_potong].reset_index(drop=True)

X_train, y_train = df_train[FITUR], df_train['target_dlog']       # target latih: perubahan log debit
X_test           = df_test[FITUR]
y_test           = df_test['target_debit_besok']                  # debit asli (m3/s) untuk evaluasi

print(f"Tanggal potong : {tgl_potong.date()}")
print(f"Data latih     : {len(df_train):,} baris ({df_train['obs_date'].min().date()} s.d. {df_train['obs_date'].max().date()})")
print(f"Data uji       : {len(df_test):,} baris ({df_test['obs_date'].min().date()} s.d. {df_test['obs_date'].max().date()})")

## TAHAP 6: Pelatihan Model (Gradient Boosting Regressor)

GBR dipilih karena membangun banyak pohon keputusan kecil secara berurutan, setiap pohon memperbaiki kesalahan pohon sebelumnya. Cocok untuk data tabular dengan hubungan non-linear seperti hujan dan debit, serta memberi nilai pentingnya fitur. Parameter dibuat cukup sederhana (pohon dangkal, `min_samples_leaf` besar) karena data hanya sekitar 1 tahun, sehingga risiko overfitting tinggi. Sebelum pelatihan akhir dilakukan validasi silang berbasis waktu (`TimeSeriesSplit`) untuk melihat kestabilan.

In [ ]:
model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.03,
    max_depth=3,
    subsample=0.8,
    min_samples_leaf=20,
    random_state=42
)

# Validasi silang berbasis waktu (data latih sudah urut tanggal)
tscv = TimeSeriesSplit(n_splits=3)
cv_mae = -cross_val_score(model, X_train, y_train, cv=tscv, scoring='neg_mean_absolute_error')
print(f"MAE (perubahan log) validasi silang per fold: {np.round(cv_mae, 3)} | rata-rata {cv_mae.mean():.3f}")

# Pelatihan akhir pada seluruh data latih
model.fit(X_train, y_train)
print("Model selesai dilatih.")

## TAHAP 7: Evaluasi Model

### 7.1 Metrik Regresi Dibanding Baseline Persistence

Prediksi model (perubahan log) diubah kembali ke m³/s lebih dulu. Baseline persistence menebak debit besok sama dengan debit hari ini. Model hanya layak dipakai bila lebih baik dari tebakan sederhana ini. Metrik: MAE (rata-rata selisih absolut, makin kecil makin baik), RMSE (lebih menghukum kesalahan besar), R² (makin dekat 1 makin baik).

In [ ]:
# Prediksi perubahan log -> kembalikan ke debit (m3/s)
pred_dlog = model.predict(X_test)
y_pred = np.clip(np.expm1(df_test['log_debit_t0'].values + pred_dlog), 0, None)
y_base = df_test['debit_t0'].values

def metrik(y, p):
    return {'MAE': mean_absolute_error(y, p),
            'RMSE': np.sqrt(mean_squared_error(y, p)),
            'R2': r2_score(y, p)}

tabel = pd.DataFrame({'Baseline Persistence': metrik(y_test, y_base),
                      'Gradient Boosting': metrik(y_test, y_pred)}).T.round(4)
display(tabel)

if tabel.loc['Gradient Boosting', 'MAE'] < tabel.loc['Baseline Persistence', 'MAE']:
    gain = (1 - tabel.loc['Gradient Boosting', 'MAE'] / tabel.loc['Baseline Persistence', 'MAE']) * 100
    print(f"GBR lebih akurat: MAE turun {gain:.1f}% dibanding baseline.")
else:
    print("GBR belum mengalahkan baseline pada MAE; pertimbangkan tuning atau fitur tambahan.")

# MAE per kota
per_kota = df_test.assign(pred=y_pred, base=y_base).groupby('kota').apply(lambda g: pd.Series({
    'MAE_GBR': mean_absolute_error(g['target_debit_besok'], g['pred']),
    'MAE_Baseline': mean_absolute_error(g['target_debit_besok'], g['base'])
})).round(3)
display(per_kota)

### 7.2 Deteksi Kondisi di Atas Ambang Siaga

Ambang siaga tiap kota = persentil ke-90 debit pada **data latih** (tidak memakai data uji agar tidak bocor). Prediksi dan kenyataan diubah menjadi kelas "Siaga" (debit ≥ ambang) atau "Normal", lalu dihitung precision, recall, dan f1. Recall pada kelas Siaga paling penting: berapa persen kejadian siaga sungguhan yang berhasil terdeteksi.

In [ ]:
ambang = df_train.groupby('kota')['debit'].quantile(0.90)
thr_test = df_test['kota'].map(ambang).values

y_true_cls = (y_test.values >= thr_test).astype(int)
y_pred_cls = (y_pred >= thr_test).astype(int)
y_base_cls = (y_base >= thr_test).astype(int)

print("=== GRADIENT BOOSTING ===")
print(classification_report(y_true_cls, y_pred_cls, target_names=['Normal', 'Siaga'], zero_division=0))
print("=== BASELINE PERSISTENCE ===")
print(classification_report(y_true_cls, y_base_cls, target_names=['Normal', 'Siaga'], zero_division=0))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, pred, judul in zip(ax, [y_pred_cls, y_base_cls], ['Gradient Boosting', 'Baseline Persistence']):
    sns.heatmap(confusion_matrix(y_true_cls, pred), annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Normal', 'Siaga'], yticklabels=['Normal', 'Siaga'], ax=a)
    a.set_title(f'Confusion Matrix: {judul}'); a.set_xlabel('Prediksi'); a.set_ylabel('Aktual')
plt.tight_layout(); plt.show()

### 7.3 Grafik Feature Importance

Menunjukkan fitur yang paling banyak mengurangi kesalahan model (menjawab bagian "fitur apa yang paling berpengaruh" pada Pertanyaan 5).

In [ ]:
imp = pd.Series(model.feature_importances_, index=FITUR).sort_values(ascending=False)

plt.figure(figsize=(9, 6))
sns.barplot(x=imp.head(15).values, y=imp.head(15).index, palette='viridis')
plt.title('15 Fitur Terpenting Model Gradient Boosting')
plt.xlabel('Tingkat kepentingan'); plt.ylabel('')
plt.tight_layout(); plt.show()

print("[JAWABAN 5]")
print(f"- Algoritma: Gradient Boosting Regressor, MAE {tabel.loc['Gradient Boosting','MAE']:.3f} vs baseline {tabel.loc['Baseline Persistence','MAE']:.3f}")
print(f"- 3 fitur teratas: {', '.join(imp.head(3).index)}")
rec = classification_report(y_true_cls, y_pred_cls, output_dict=True, zero_division=0)['1']['recall']
print(f"- Recall deteksi Siaga: {rec:.2f}")

## TAHAP 8: Menyimpan Artefak untuk Dashboard Streamlit

Semua yang dibutuhkan dashboard disimpan ke folder `artifacts_banjir/` lalu di-zip:

| File | Fungsi di dashboard |
|:---|:---|
| `model_gbr_debit.joblib` | Model terlatih untuk prediksi debit besok |
| `model_meta.json` | Daftar fitur (urutan harus sama), rumus mengembalikan prediksi ke m³/s, daftar kota, ambang Waspada/Siaga/Awas per kota, metrik evaluasi, importance fitur |
| `riwayat_harian.csv` | Data harian bersih (kota, tanggal, debit, hujan, koordinat bila ada) untuk grafik dan untuk mengambil riwayat lag |
| `ringkasan_eda.csv` | Tabel risiko per kota dari Grafik 2 untuk halaman ringkasan |

Ambang status: **Waspada** = P75, **Siaga** = P90, **Awas** = P95 dari debit data latih per kota.

In [ ]:
OUT_DIR = '/content/drive/MyDrive/artifacts_banjir' if os.path.exists('/content/drive/MyDrive') else 'artifacts_banjir'
os.makedirs(OUT_DIR, exist_ok=True)

# 1. Model
joblib.dump(model, os.path.join(OUT_DIR, 'model_gbr_debit.joblib'))

# 2. Metadata
ambang_df = df_train.groupby('kota')['debit'].quantile([0.75, 0.90, 0.95]).unstack()
ambang_df.columns = ['waspada_p75', 'siaga_p90', 'awas_p95']

meta = {
    'target': 'debit_besok',
    'transformasi_target': 'model memprediksi dlog = log1p(debit_besok) - log1p(debit_hari_ini); debit_besok = expm1(log1p(debit_hari_ini) + dlog)',
    'fitur_urut': FITUR,
    'fitur_numerik': FITUR_NUMERIK,
    'daftar_kota': sorted(df['kota'].unique().tolist()),
    'ambang_per_kota': ambang_df.round(4).to_dict(orient='index'),
    'metrik_uji': {
        'GBR': {k: float(v) for k, v in tabel.loc['Gradient Boosting'].items()},
        'Baseline': {k: float(v) for k, v in tabel.loc['Baseline Persistence'].items()},
        'recall_siaga': float(rec)
    },
    'feature_importance': {k: float(v) for k, v in imp.items()},
    'tanggal_potong_split': str(tgl_potong.date()),
    'tanggal_data_terakhir': str(df['obs_date'].max().date()),
    'catatan': 'Fitur dibuat oleh buat_fitur(); dashboard wajib memakai logika sama. Satuan debit m3/s, hujan mm/hari.'
}
with open(os.path.join(OUT_DIR, 'model_meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

# 3. Riwayat harian
kolom_riwayat = [c for c in ['location_id', 'kota', 'river_name', 'province', 'obs_date', 'debit', 'hujan',
                             'latitude', 'longitude', 'elevation_m', 'population'] if c in df.columns]
df[kolom_riwayat].to_csv(os.path.join(OUT_DIR, 'riwayat_harian.csv'), index=False)

# 4. Ringkasan EDA
risiko.to_csv(os.path.join(OUT_DIR, 'ringkasan_eda.csv'))

# 5. Uji muat ulang model (memastikan artefak valid)
model_cek = joblib.load(os.path.join(OUT_DIR, 'model_gbr_debit.joblib'))
selisih = np.abs(model_cek.predict(X_test.head(5)) - model.predict(X_test.head(5))).max()
print(f"Uji muat ulang model: selisih prediksi maksimum = {selisih:.6f} (harus 0)")

# 6. Zip untuk diunduh
with zipfile.ZipFile('artifacts_banjir.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for nama in os.listdir(OUT_DIR):
        z.write(os.path.join(OUT_DIR, nama), arcname=nama)

print("File tersimpan:", os.listdir(OUT_DIR))
try:
    from google.colab import files
    files.download('artifacts_banjir.zip')
except Exception:
    print("Bukan Colab: ambil artifacts_banjir.zip dari folder kerja.")